In [19]:
import re
import string

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # for tab tokenization
nltk.download('averaged_perceptron_tagger_eng') # for POS tagging

[nltk_data] Downloading package punkt to /Users/bryan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/bryan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/bryan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/bryan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/bryan/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [20]:
from statsmodels.stats.inter_rater import fleiss_kappa

# Step 1: Map labels to numeric (if they are not already)
# For example, if labels are 'Positive', 'Negative', 'Neutral', etc.
df = pd.read_csv("annotated_cleaned.csv")
label_mapping = {label: idx for idx, label in enumerate(pd.unique(df[['label1', 'label2', 'label3']].values.ravel()))}
df_encoded = df[['label1', 'label2', 'label3']].replace(label_mapping)

# Step 2: Build a table of counts per label per item
# For each row, count how many times each label appears among Review1-3
labels = list(label_mapping.values())
rating_matrix = []

for _, row in df_encoded.iterrows():
    counts = [sum(row == label) for label in labels]
    rating_matrix.append(counts)

rating_matrix = np.array(rating_matrix)

# Step 3: Apply Fleiss' Kappa
kappa = fleiss_kappa(rating_matrix)
print(f"🧠 Fleiss' Kappa: {kappa:.4f}")


/var/folders/lk/5c46yk7x18j_h8wp31rl548w0000gn/T/ipykernel_73606/2873338233.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df[['label1', 'label2', 'label3']].replace(label_mapping)


🧠 Fleiss' Kappa: 0.8886


In [21]:
df = pd.read_csv('annotated_cleaned.csv')
# give me the number of positive, negative, and neutral labels
df['Final Sentiment'].value_counts()

Final Sentiment
neutral     420
negative    416
positive    413
Name: count, dtype: int64

In [22]:
stop_words = set(stopwords.words('english'))
tokenizer = TweetTokenizer(strip_handles=True, reduce_len=True)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Microtext + Lemmatization
def preprocess_microtext(text):
  if not isinstance(text, str):
    text = str(text)
  # Normalization
  # Step 1: Lowercasing
  text = text.lower()

  # Step 2: Remove URLs
  text = re.sub(r"http\S+|www\S+|https\S+", '', text)

  # Step 3: Remove Mentions (e.g., @username)
  text = re.sub(r'@\w+', '', text)

  # Step 4: Remove Hashtags (Optional, or you can keep hashtags as keywords)
  text = re.sub(r'#\w+', '', text)

  # Step 5: Tokenize
  tokens = tokenizer.tokenize(text)

  # Step 6: Remove Punctuation and Stopwords
  tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]

  # Step 7: Stemming (optional)
  # tokens = [stemmer.stem(word) for word in tokens]  # Uncomment if you want to apply stemming

  # Step 8: Lemmatization
  tokens = [lemmatizer.lemmatize(word) for word in tokens]

  return " ".join(tokens)

In [23]:
df = pd.read_csv('annotated_cleaned.csv')
df['review_text'] = df['review_text'].apply(preprocess_microtext)
df.to_csv('annotated_microtext_lemma.csv', index=False)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1249 entries, 0 to 1248
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Platform         1249 non-null   object
 1   Game             1249 non-null   object
 2   review_text      1249 non-null   object
 3   label1           1249 non-null   object
 4   label2           1249 non-null   object
 5   label3           1249 non-null   object
 6   Final Sentiment  1249 non-null   object
dtypes: object(7)
memory usage: 68.4+ KB


In [25]:
df["Final Sentiment"].value_counts()

Final Sentiment
neutral     420
negative    416
positive    413
Name: count, dtype: int64